In [2]:
import pandas as pd
import requests
import numpy as np
from datetime import datetime
import time
import os
import logging
from pathlib import Path

In [3]:
url = "https://transparency.entsog.eu/api/v1/operatorpointdirections"

params = {
    "limit": -1,
    "tSOCountry": "BE",
    "hasData": 1,
}

resp = requests.get(url, params=params)
df = pd.DataFrame(resp.json()["operatorpointdirections"])
df["pointId"] = (df["operatorKey"] + df["pointKey"] + df["directionKey"]).str.lower()
df.to_csv("Operator_Directions/Operator_point_directions_belgium.csv")

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("ingestion_entsog.log", encoding="utf-8"),
        logging.StreamHandler(),          # affiche aussi dans la console
    ],
)
log = logging.getLogger(__name__)

In [5]:
url = "https://transparency.entsog.eu/api/v1/operationaldatas"

operator_points = pd.read_csv(r"C:\Users\PC\Documents\Projets\TFE\Ingestion\Operator_Directions\Operator_point_directions_belgium.csv")
all_year = np.arange(2021,datetime.now().year+1)

for i in range (len(all_year)):
    metriques = [] 
    year = all_year[i]
    path_year = rf'C:\Users\PC\Documents\Projets\TFE\Ingestion\Operational_datas\Belgium\{year}'
    if not os.path.exists(path_year): os.makedirs(path_year)

    for j in range(len(operator_points)):
        point_name = operator_points["pointKey"][j] + '_' + operator_points["directionKey"][j]
        point_id = operator_points["pointId"][j]
        print(point_name)
        print(point_id)
        from_date = f"{year}-01-01"
        if year == datetime.now().year:
            to_date = datetime.now().strftime("%Y-%m-%d")
        else : 
            to_date = f"{year}-12-31"

        parameters = {
        "indicator": "Physical Flow",
        "pointDirection": point_id,
        "from": from_date,
        "to": to_date,
        "periodType": "hour",
        "timezone": "CET",
        "limit": -1,
        }
        debut = time.perf_counter()
        try:
            reponse = requests.get(url, params=parameters)
            reponse.raise_for_status()
        except requests.exceptions.RequestException as e:
            log.error(f"{year} | {point_name} | échec : {e}")
            metriques.append({"annee": year, "point": point_name, "lignes": 0,
                                              "taille_ko": 0, "duree_s": 0})
            continue

        df = pd.DataFrame(reponse.json()["operationaldatas"])
        fichier = f"{path_year}/{point_name}.csv"
        df.to_csv(fichier, index=False)

        duree = time.perf_counter() - debut
        taille_ko = os.path.getsize(fichier) / 1024
        log.info(f"{year} | {point_name} | {len(df)} lignes | {taille_ko:.0f} Ko | {duree:.2f} s")
        metriques.append({"annee": year, "point": point_name, "lignes": len(df),
                  "taille_ko": taille_ko, "duree_s": duree})
    
    pd.DataFrame(metriques).to_csv(f"{path_year}/metriques_ingestion.csv", index=False)

DIS-00191_entry
be-tso-0001dis-00191entry


2026-10-02 17:55:32,445 | INFO | 2021 | DIS-00191_entry | 8598 lignes | 3058 Ko | 32.92 s


DIS-00192_entry
be-tso-0001dis-00192entry


2026-10-02 17:55:40,567 | INFO | 2021 | DIS-00192_entry | 6693 lignes | 2380 Ko | 8.12 s


FNC-00033_entry
be-tso-0001fnc-00033entry


...

2026-10-02 18:44:50,894 | INFO | 2026 | LNG-00017_exit | 6575 lignes | 2562 Ko | 7.43 s


UGS-00002_exit
be-tso-0001ugs-00002exit


2026-10-02 18:44:57,726 | INFO | 2026 | UGS-00002_exit | 6575 lignes | 2314 Ko | 6.83 s


VTP-00029_exit
be-tso-0001vtp-00029exit


2026-10-02 18:45:00,002 | ERROR | 2026 | VTP-00029_exit | échec : 404 Client Error: Not Found for url: https://transparency.entsog.eu/api/v1/operationaldatas?indicator=Physical+Flow&pointDirection=be-tso-0001vtp-00029exit&from=2026-01-01&to=2026-10-02&periodType=hour&timezone=CET&limit=-1
